# Credit Card Fraud Detection
Train a baseline classifier from `credit_card_fraud_custom.csv`, evaluate it, and download the trained model for your Streamlit app.

**Before running:** upload `credit_card_fraud_custom.csv` to the Colab session using the file sidebar, or use the upload cell below.

In [ ]:
!pip -q install pandas scikit-learn joblib

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select credit_card_fraud_custom.csv

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("credit_card_fraud_custom.csv")
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nTarget distribution:")
print(df["IsFraud"].value_counts(dropna=False))

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y = pd.to_numeric(df["IsFraud"], errors="raise").astype(int)
X = df.drop(columns=["IsFraud", "TransactionID"], errors="ignore").copy()

dt = pd.to_datetime(X["TransactionDate"], errors="coerce")
X["TransactionHour"] = dt.dt.hour
X["TransactionDayOfWeek"] = dt.dt.dayofweek
X["TransactionMonth"] = dt.dt.month
X = X.drop(columns=["TransactionDate"])

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250, class_weight="balanced", random_state=42, n_jobs=-1
    )),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, preds))
print("\nClassification report:")
print(classification_report(y_test, preds, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.4f}")

In [ ]:
MODEL_PATH = "fraud_model.joblib"
joblib.dump(model, MODEL_PATH)
print("Saved:", MODEL_PATH)

In [ ]:
from google.colab import files
files.download("fraud_model.joblib")

## Next steps
1. Download `fraud_model.joblib` from the last cell.
2. Upload it to the root of your GitHub repository alongside `app.py` and `requirements.txt`.
3. Deploy `app.py` using Streamlit Community Cloud.

Keep the dataset and model private if they contain sensitive or licensed data. This starter model is educational only; do not use it as the sole basis for real fraud decisions.